###Author - Haaris Khalil
###Course - Data Management

###Week 2- Exercise – Individual - Data Quality Audit

This dataset contains 11 records of customer financial transactions, tracking elements like customer IDs, transaction dates, payment methods, currency, region, and transaction status. Its primary business use case is likely for revenue reporting, customer behavior analytics, and payment processing audits. High data quality is critical here, because incorrect amounts or mismatched currencies could severely distort financial reports.

In [10]:
import pandas as pd
import numpy as np

# Load the dataset (Make sure the CSV is uploaded to your Colab session)
df = pd.read_csv('week2_customer_transactions_messy.csv')

print("Data loaded successfully! Shape:", df.shape)

Data loaded successfully! Shape: (11, 9)


In [11]:
issue_table = pd.DataFrame([
    ['Missing values (customer_id, amount, payment_method, last_updated)', 'Completeness', 'Hinders automated processing and revenue tracking.'],
    ['Duplicate row (T0006)', 'Uniqueness', 'Inflates transaction volume and revenue artificially.'],
    ['Negative or zero amounts, Impossible date (Feb 30)', 'Validity', 'Corrupts financial metrics and downstream analytics.'],
    ['Mixed date formats, inconsistent capitalization (EUR/EURO)', 'Consistency', 'Prevents accurate grouping and aggregation.'],
    ['Transaction date (Feb 30) after last_updated date (Feb 15)', 'Integrity', 'Violates logical temporal sequence.']
], columns=['Issue', 'Dimension', 'Business Impact'])

display(issue_table)

,Issue,Dimension,Business Impact
0,"Missing values (customer_id, amount, payment_m...",Completeness,Hinders automated processing and revenue track...
1,Duplicate row (T0006),Uniqueness,Inflates transaction volume and revenue artifi...
2,"Negative or zero amounts, Impossible date (Feb...",Validity,Corrupts financial metrics and downstream anal...
3,"Mixed date formats, inconsistent capitalizatio...",Consistency,Prevents accurate grouping and aggregation.
4,Transaction date (Feb 30) after last_updated d...,Integrity,Violates logical temporal sequence.


In [12]:
total_cells = df.shape[0] * df.shape[1]

kpis = {}
kpis['Completeness Rate'] = 1 - (df.isna().sum().sum() / total_cells)
kpis['Duplication Rate'] = df.duplicated().sum() / df.shape[0]
kpis['Valid Amount Rate'] = (df['amount'] > 0).sum() / df.shape[0]

# Formatting for display
kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Value'])
kpi_df['Value'] = (kpi_df['Value'] * 100).round(1).astype(str) + '%'
display(kpi_df)

,KPI,Value
0,Completeness Rate,96.0%
1,Duplication Rate,9.1%
2,Valid Amount Rate,72.7%


* **Completeness Rate (~96.0%):** Most of the data cells are filled, but the scattered null values across critical columns (like IDs and amounts) mean rows cannot be reliably processed in automated pipelines.
* **Duplication Rate (~9.1%):** Almost 10% of the dataset consists of duplicated entries. In a larger dataset, this would artificially inflate revenue metrics.
* **Valid Amount Rate (~72.7%):** Over a quarter of the transactions have an amount that is missing, zero, or negative. This is a severe issue for a financial dataset.

In [13]:
rules = {
    'missing_data_any_column': df.isnull().any(axis=1),
    'exact_row_duplicates': df.duplicated(keep='first'),
    'invalid_or_negative_amounts': (df['amount'] <= 0) | df['amount'].isnull(),
    'inconsistent_text_formatting': df['currency'].isin(['EURO']) | df['region'].str.islower().fillna(False)
}

print("Validation rules defined successfully.")

Validation rules defined successfully.


In [14]:
audit_summary = pd.DataFrame([
    ['Missing Data', 4, 'High', 'Investigate source system for missing IDs; impute or drop missing categorical data.'],
    ['Invalid Amounts', 3, 'High', 'Quarantine records with negative or null amounts for manual review.'],
    ['Formatting Issues', 2, 'Low', 'Standardize categorical columns to uppercase text using string methods.'],
    ['Duplicates', 1, 'Medium', 'Drop exact duplicate rows keeping the first occurrence.']
], columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])

display(audit_summary)

,issue_type,affected_rows,severity,recommended_next_action
0,Missing Data,4,High,Investigate source system for missing IDs; imp...
1,Invalid Amounts,3,High,Quarantine records with negative or null amoun...
2,Formatting Issues,2,Low,Standardize categorical columns to uppercase t...
3,Duplicates,1,Medium,Drop exact duplicate rows keeping the first oc...


* **Recommendation 1:** Apply `df.drop_duplicates()` to remove identical rows safely.
* **Recommendation 2:** Apply `.str.upper()` and `.str.strip()` to the `currency`, `payment_method`, and `region` columns to resolve case and spacing inconsistencies.
* **Recommendation 3:** Coerce all date columns to standard datetime objects using `pd.to_datetime(..., errors='coerce')` so that invalid dates become `NaT` (Not a Time), making them easier to filter.

1. **Which KPI gave the strongest signal?** The Valid Amount Rate gave the strongest signal because it highlighted that over a quarter of the transactions had zero, negative, or missing amounts, which is catastrophic for financial data.
2. **Which issue should be escalated?** The missing `customer_id` and the invalid negative amounts should be escalated immediately, as they directly prevent accurate revenue reporting and customer tracking.

In [15]:
def summarize_rule_violations(rule_dictionary):
    """Summarize affected row counts for each validation rule."""
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

# Example usage with your `rules` dictionary:
display(summarize_rule_violations(rules))

,rule_name,affected_rows
0,missing_data_any_column,3
2,invalid_or_negative_amounts,3
3,inconsistent_text_formatting,2
1,exact_row_duplicates,1


* **What each parameter means:** The `rule_dictionary` parameter accepts a Python dictionary where the keys are descriptive rule names (strings) and the values are boolean arrays (pandas Series) flagging which rows violate that specific rule.
* **Why you selected default values:** The rules were selected based on standard e-commerce logic to capture fundamental errors without flagging legitimate edge cases.
* **How changing parameters affects KPI/audit results:** Changing these parameters to be stricter (for example, flagging amounts over 500 as anomalies) would increase the number of affected rows, thereby lowering the valid amount KPI and requiring more manual data review.